In [2]:
import os
import pandas as pd
from datetime import datetime
import importlib
import numpy as np

np.set_printoptions(suppress=True)

In [3]:
# Relative imports
os.chdir("..")
import hidden_state_model.processor
importlib.reload(hidden_state_model.processor)
Processor = hidden_state_model.processor.Processor
os.chdir("hidden_state_model")

### Read (and compact) dataframes

In [4]:
compact = True

In [5]:
# Iterate over files in dfs/*.parquet and combine to one df
dfs = []

read = []
for file in os.listdir("data"):
    if file.endswith(".parquet"):
        read.append(file)
        print(f"Reading {file}")
        df = pd.read_parquet(f"data/{file}")
        dfs.append(df)
    if file.endswith(".csv"):
        read.append(file)
        df = pd.read_csv(f"data/{file}", index_col=0)
        dfs.append(df)

raw_df = pd.concat(dfs)

if compact and len(dfs) > 10:
    print("Compacintg dfs")

    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    raw_df.to_parquet(f"data/compacted_{timestamp}.parquet")
    
    # Move read files to trash and write combined df to dfs/combined_{timestamp}.parquet
    trash = "data/trash"
    for f in read:
        os.rename(f"data/{f}", f"{trash}/{f}")

dfs = []  # Clear memory
raw_df

Reading combined.parquet


,prev_entry,public_cards,player_piles,current_player_i,bet_in_stage,bet_in_game,player_has_played,player_is_folded,first_better_i,big_blind,player_name,player_type,action,amount,p,relative_ev,rank,tiebreakers,hand_index,state_id
state_id,,,,,,,,,,,,,,,,,,,,
32e30c0a-821c-4bcb-9986-2844eb402812,None,[],"[98, 96]",0,"[2, 4]","[2, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,call,2,0.3953,0.011859,0,"[2, 1, 0, 0, 0]",NaN,NaN
b7ec800e-1e76-4d7f-9221-9a9602769945,32e30c0a-821c-4bcb-9986-2844eb402812,"[31, 15, 11]","[96, 96]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.5987,0.023948,1,"[2, 11, 5, 1, 0]",NaN,NaN
7fa95b11-9edb-4c33-810e-dafe3e297db9,b7ec800e-1e76-4d7f-9221-9a9602769945,"[31, 15, 11]","[92, 88]",0,"[4, 8]","[8, 12]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,4,0.5987,0.059870,1,"[2, 11, 5, 1, 0]",NaN,NaN
e93561e9-938c-4140-a5d7-35eb2946f461,7fa95b11-9edb-4c33-810e-dafe3e297db9,"[31, 15, 11, 50]","[88, 88]",0,"[0, 0]","[12, 12]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,check,0,0.5906,0.070872,2,"[11, 2, 5, 0, 0]",NaN,NaN
a2683c14-bd1c-431e-9cf1-5486dcc70d4b,e93561e9-938c-4140-a5d7-35eb2946f461,"[31, 15, 11, 50]","[88, 77]",0,"[0, 11]","[12, 23]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,11,0.5906,0.103355,2,"[11, 2, 5, 0, 0]",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,6127529072,"[8, 9, 36]","[152, 40]",0,"[0, 0]","[4, 4]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.6240,0.024960,1,"[8, 10, 9, 4, 0]",NaN,NaN
6431104400,6431248928,"[8, 9, 36, 6]","[148, 36]",0,"[0, 0]","[8, 8]","[False, False]","[False, False]",0,4,Tord,HumanPlayer,raise,4,0.5459,0.043672,1,"[8, 10, 9, 6, 0]",NaN,NaN
6127537840,6431104400,"[8, 9, 36, 6]","[144, 26]",0,"[4, 10]","[12, 18]","[True, True]","[False, False]",0,4,Tord,HumanPlayer,call,6,0.5459,0.081885,1,"[8, 10, 9, 6, 0]",NaN,NaN


In [7]:
raw_df.dtypes

prev_entry            object
public_cards          object
player_piles          object
current_player_i       int64
bet_in_stage          object
bet_in_game           object
player_has_played     object
player_is_folded      object
first_better_i         int64
big_blind              int64
player_name           object
player_type           object
action                object
amount                 int64
p                    float64
relative_ev          float64
rank                   int64
tiebreakers           object
hand_index           float64
state_id             float64
dtype: object

In [8]:
# Check for conflicting rows
dupe_df = raw_df[raw_df.index.duplicated()]
assert len(dupe_df) == 0, dupe_df

## Process data

In [9]:
processor = Processor(raw_df)
df = processor.get_processed_df()
df

Cannot process state without parent prev_entry               6159592272.0
public_cards             [30, 34, 50]
player_piles                [92, 100]
current_player_i                    0
bet_in_stage                   [0, 0]
bet_in_game                    [4, 4]
player_has_played       [False, True]
player_is_folded       [False, False]
first_better_i                      1
big_blind                           4
player_name                      Tord
player_type               HumanPlayer
action                          check
amount                              0
p                              0.2599
relative_ev                  0.010396
rank                                0
tiebreakers          [11, 8, 4, 2, 1]
hand_index                        NaN
state_id                          NaN
Name: 6138187200, dtype: object
Cannot process state without parent prev_entry               6138187200.0
public_cards         [30, 34, 50, 18]
player_piles                [92, 100]
current_player_i      

,game_id,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,...,opponent_check_river,opponent_check_showdown,action,amount,excess_rank,p,relative_ev,stage,player_name,n_players
32e30c0a-821c-4bcb-9986-2844eb402812,6f6dc43d-8d27-4998-9e87-5254bf984425,0,0,0,0,0,0,0,0,0,...,0,0,call,2,0,0.3953,0.011859,preflop,Tord,2
b7ec800e-1e76-4d7f-9221-9a9602769945,6f6dc43d-8d27-4998-9e87-5254bf984425,0,0,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.5987,0.023948,flop,Tord,2
7fa95b11-9edb-4c33-810e-dafe3e297db9,6f6dc43d-8d27-4998-9e87-5254bf984425,0,4,0,0,0,2,0,0,0,...,0,0,call,4,1,0.5987,0.059870,flop,Tord,2
e93561e9-938c-4140-a5d7-35eb2946f461,6f6dc43d-8d27-4998-9e87-5254bf984425,0,4,0,0,0,2,4,0,0,...,0,0,check,0,1,0.5906,0.070872,turn,Tord,2
a2683c14-bd1c-431e-9cf1-5486dcc70d4b,6f6dc43d-8d27-4998-9e87-5254bf984425,0,4,0,0,0,2,4,0,0,...,0,0,call,11,1,0.5906,0.103355,turn,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,c95d1c0b-151f-4afd-acdc-f310472c56d5,0,0,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.6240,0.024960,flop,Tord,2
6431104400,c95d1c0b-151f-4afd-acdc-f310472c56d5,0,4,0,0,0,2,0,0,0,...,0,0,raise,4,1,0.5459,0.043672,turn,Tord,2
6127537840,c95d1c0b-151f-4afd-acdc-f310472c56d5,0,4,4,0,0,2,0,0,0,...,0,0,call,6,1,0.5459,0.081885,turn,Tord,2
6431250896,c95d1c0b-151f-4afd-acdc-f310472c56d5,0,4,4,0,0,2,0,6,0,...,1,0,check,0,0,0.9412,0.169416,river,Tord,2


In [10]:
df.dtypes

game_id                     object
raise_preflop                int64
raise_flop                   int64
raise_turn                   int64
raise_river                  int64
raise_showdown               int64
call_preflop                 int64
call_flop                    int64
call_turn                    int64
call_river                   int64
call_showdown                int64
check_preflop                int64
check_flop                   int64
check_turn                   int64
check_river                  int64
check_showdown               int64
opponent_raise_preflop       int64
opponent_raise_flop          int64
opponent_raise_turn          int64
opponent_raise_river         int64
opponent_raise_showdown      int64
opponent_call_preflop        int64
opponent_call_flop           int64
opponent_call_turn           int64
opponent_call_river          int64
opponent_call_showdown       int64
opponent_check_preflop       int64
opponent_check_flop          int64
opponent_check_turn 

## Training

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [12]:
X = df.drop(["game_id", "action", "amount"], axis=1)
y = df["action"]
groups = df["game_id"]  # Group by 'game_id' to ensure no data leakage

In [13]:
X

,raise_preflop,raise_flop,raise_turn,raise_river,raise_showdown,call_preflop,call_flop,call_turn,call_river,call_showdown,...,opponent_check_flop,opponent_check_turn,opponent_check_river,opponent_check_showdown,excess_rank,p,relative_ev,stage,player_name,n_players
32e30c0a-821c-4bcb-9986-2844eb402812,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0.3953,0.011859,preflop,Tord,2
b7ec800e-1e76-4d7f-9221-9a9602769945,0,0,0,0,0,2,0,0,0,0,...,1,0,0,0,1,0.5987,0.023948,flop,Tord,2
7fa95b11-9edb-4c33-810e-dafe3e297db9,0,4,0,0,0,2,0,0,0,0,...,1,0,0,0,1,0.5987,0.059870,flop,Tord,2
e93561e9-938c-4140-a5d7-35eb2946f461,0,4,0,0,0,2,4,0,0,0,...,1,1,0,0,1,0.5906,0.070872,turn,Tord,2
a2683c14-bd1c-431e-9cf1-5486dcc70d4b,0,4,0,0,0,2,4,0,0,0,...,1,1,0,0,1,0.5906,0.103355,turn,Tord,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431248928,0,0,0,0,0,2,0,0,0,0,...,1,0,0,0,1,0.6240,0.024960,flop,Tord,2
6431104400,0,4,0,0,0,2,0,0,0,0,...,1,0,0,0,1,0.5459,0.043672,turn,Tord,2
6127537840,0,4,4,0,0,2,0,0,0,0,...,1,0,0,0,1,0.5459,0.081885,turn,Tord,2
6431250896,0,4,4,0,0,2,0,6,0,0,...,1,0,1,0,0,0.9412,0.169416,river,Tord,2


In [14]:
y

32e30c0a-821c-4bcb-9986-2844eb402812     call
b7ec800e-1e76-4d7f-9221-9a9602769945    raise
7fa95b11-9edb-4c33-810e-dafe3e297db9     call
e93561e9-938c-4140-a5d7-35eb2946f461    check
a2683c14-bd1c-431e-9cf1-5486dcc70d4b     call
                                        ...  
6431248928                              raise
6431104400                              raise
6127537840                               call
6431250896                              check
6431104448                               fold
Name: action, Length: 3993, dtype: object

In [15]:
# Identify categorical columns (excluding 'game_id')
categorical_cols = ["excess_rank", "stage", "player_name"]

# Preprocessing pipeline: OneHotEncoding for categorical and scaling for numerical
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(drop="first"), categorical_cols)],
    remainder="passthrough",
)

# Create the full pipeline with logistic regression
model = Pipeline(
    [
        ("preprocess", preprocessor),
        (
            "classifier",
            LogisticRegression(
                multi_class="multinomial", solver="lbfgs", max_iter=10_000
            ),
        ),
    ]
)

In [16]:
# Grouped train-test split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (3237, 36)
Test shape: (756, 36)


In [17]:
# Train the model
model.fit(X_train, y_train)

# Evaluate the model
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.5952380952380952


In [18]:
probabilities = model.predict_proba(X_test)
prob_df = pd.DataFrame(probabilities, columns=model.classes_, index=X_test.index)
prob_df["true"] = y_test.values
prob_df["pred"] = model.predict(X_test)
prob_df["correct"] = prob_df["true"] == prob_df["pred"]
prob_df["goodness"] = prob_df.apply(lambda x: x.get(x["true"]) or 0, axis=1)
print("Accuracy", prob_df["correct"].mean())
print("Mean goodness: ", prob_df["goodness"].mean())
prob_df

Accuracy 0.5952380952380952
Mean goodness:  0.5080744385710847


,call,check,fold,raise,true,pred,correct,goodness
6159628224,0.399417,0.339652,0.152734,0.108197,fold,call,False,0.152734
8bf54d28-d4f3-4a91-ac69-c92c5f6ceead,0.402500,0.350429,0.131342,0.115729,call,call,True,0.402500
139754fa-4929-4549-b3fc-e8af48ddafe0,0.618515,0.183355,0.124605,0.073526,call,call,True,0.618515
521b246e-8adc-4179-9c38-e93cead92475,0.043702,0.468259,0.000481,0.487558,raise,raise,True,0.487558
a0f719e8-3a79-4e5a-8574-7ca49e6f200e,0.013332,0.632973,0.000007,0.353689,raise,check,False,0.353689
...,...,...,...,...,...,...,...,...
6102526320,0.310904,0.399023,0.176338,0.113735,check,check,True,0.399023
6127312464,0.195697,0.128986,0.034715,0.640602,check,raise,False,0.128986
6127438960,0.130502,0.192648,0.032012,0.644838,check,raise,False,0.192648
6431194992,0.404502,0.207195,0.043521,0.344782,raise,call,False,0.344782


In [19]:
# Look at incorrect predictions
print(prob_df[prob_df["correct"] == False].shape[0], "incorrect predictions:")
prob_df[prob_df["correct"] == False]

306 incorrect predictions:


,call,check,fold,raise,true,pred,correct,goodness
6159628224,0.399417,0.339652,0.152734,0.108197,fold,call,False,0.152734
a0f719e8-3a79-4e5a-8574-7ca49e6f200e,0.013332,0.632973,0.000007,0.353689,raise,check,False,0.353689
aa104b52-4240-47b4-b990-75807eddf472,0.404908,0.348094,0.130388,0.116610,check,call,False,0.348094
7727185c-9a18-4a3c-8f2a-dafb25730980,0.418602,0.346853,0.103881,0.130664,check,call,False,0.346853
41498704-7eca-4076-bdf5-6a722fb600e2,0.426569,0.346262,0.083396,0.143773,raise,call,False,0.143773
...,...,...,...,...,...,...,...,...
6127526000,0.539013,0.120278,0.302777,0.037933,fold,call,False,0.302777
6127534288,0.327868,0.323350,0.278847,0.069936,check,call,False,0.323350
6127312464,0.195697,0.128986,0.034715,0.640602,check,raise,False,0.128986
6127438960,0.130502,0.192648,0.032012,0.644838,check,raise,False,0.192648
